In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings
warnings.filterwarnings("ignore") #suppress warnings in outputs

# EDA

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt 

data = pd.read_csv("../input/s-and-p-500-spy/spy.csv")

print(f"The shape of the data is: {data.shape}\n")
print(data.columns)
print(f"\n {data.dtypes} \n")
data.head(3)

In [ ]:
#null/nan check 
print(f"check any null: {data.isnull().any().any()}")
print(f"check any NaN: {data.isna().any().any()}")

In [ ]:
print(f"unique values of 'Day': {sorted(data['Day'].unique())}; \n num of 'Day': {len(data['Day'].unique())}")
print(f"\n unique values of 'Weekday': {sorted(data['Weekday'].unique())}; \n num of 'Weekday': {len(data['Weekday'].unique())}")
print(f"\n unique values of 'Week': {sorted(data['Week'].unique())}; \n num of 'Week': {len(data['Week'].unique())}")
print(f"\n unique values of 'Month': {sorted(data['Month'].unique())}; \n num of 'Month': {len(data['Month'].unique())}")
print(f"\n unique values of 'Year': {sorted(data['Year'].unique())}; \n num of 'Year': {len(data['Year'].unique())}")

In [ ]:
#only select the data starting 2020 and afterward
data = data[data["Year"] >= 2020]
print(f"\n unique values of 'Year': {sorted(data['Year'].unique())}; \n num of 'Year': {len(data['Year'].unique())}")

In [ ]:
#log the prices and the vol
data.loc[:, ["Open", "High", "Low", "Close", "Volume"]] = data.loc[:, ["Open", "High", "Low", "Close", "Volume"]].apply(np.log)
print(f'check any inf after log: {data.loc[:, ["Open", "High", "Low", "Close", "Volume"]].apply(np.isinf).any().any()} \n')
print(f"\n {data.dtypes} \n")
data.head(3)

In [ ]:
data_plot = data.drop(columns = ["Date", "Day", "Weekday", "Week", "Month", "Year"], axis = 1)
data_plot = data_plot.apply(lambda x: (x - x.mean())/x.std()) #global normalization only for visualization

grid = sns.PairGrid(data_plot)
grid.map_upper(sns.scatterplot)
grid.map_lower(sns.kdeplot, fill=True)
grid.map_diag(sns.histplot, kde=True)

In [ ]:
# Determine the number of rows and columns for the subplot grid
num_cols = len(data_plot.columns)
num_rows = int(np.ceil(np.sqrt(num_cols)))
num_cols_per_row = int(np.ceil(num_cols / num_rows))

# Create subplots with the calculated layout
fig, axes = plt.subplots(num_rows, num_cols_per_row, figsize=(15, 10)) 

# Iterate through columns and create boxplots
axes = axes.flatten() # Flatten axes for easy iteration
for i, column in enumerate(data_plot.columns):
    ax = axes[i]
    ax.boxplot(data[column])
    ax.set_title(column)

# Remove empty subplots (if any)
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

* ***add the return columns for each real column***

In [ ]:
#add log price returns
# Step 1: Select numeric columns to compute differences
numeric_cols = data.select_dtypes(include=['float64']).columns #prices are all in 'float'

# Step 2: Calculate row-wise differences for numeric columns
diff_df = data[numeric_cols].diff()

# Step 3: Add differences to the original DataFrame (optional)
# Prefix new columns with 'diff_'
data[[f'diff_{col}' for col in numeric_cols]] = diff_df

# Step 4: Print results
print("DataFrame with differences:\n", data.head(5))

# Step 5: Debug - Check for infinite or NaN values in differences
#print("\n Any infinite values in differences:", data.isin([np.inf, -np.inf]).any().any())
print("\n Any infinite values in differences:", data.drop("Date", axis=1).apply(np.isinf).any().any())
print("Any NaN values in differences:", data.isna().any().any())

* ***categorizing the calendar features***

In [ ]:
#turn the calendar features into category
data.loc[:, ["Day", "Weekday", "Week", "Month", "Year"]] = data.loc[:, ["Day", "Weekday", "Week", "Month", "Year"]].astype(str).astype("category")
print("Updated data types:\n", data.dtypes)

* ***table of 1-day holding period return***

In [ ]:
d_hpr = data.drop(columns = ['Open', 'High', 'Low', 'Close', 'Volume'], axis = 1)
d_hpr = d_hpr.dropna()

print(f"the shape of 1d_hpr is: {d_hpr.shape}\n")
d_hpr.head(3)

In [ ]:
hpr_plot = d_hpr.drop(columns = ["Date", "Day", "Weekday", "Week", "Month", "Year"], axis = 1)

grid = sns.PairGrid(hpr_plot)
grid.map_upper(sns.scatterplot)
grid.map_lower(sns.kdeplot, fill=True)
grid.map_diag(sns.histplot, kde=True)

In [ ]:
# Determine the number of rows and columns for the subplot grid
num_cols = len(hpr_plot.columns)
num_rows = int(np.ceil(np.sqrt(num_cols)))
num_cols_per_row = int(np.ceil(num_cols / num_rows))

# Create subplots with the calculated layout
fig, axes = plt.subplots(num_rows, num_cols_per_row, figsize=(15, 10)) 

# Iterate through columns and create boxplots
axes = axes.flatten() # Flatten axes for easy iteration
for i, column in enumerate(hpr_plot.columns):
    ax = axes[i]
    ax.boxplot(d_hpr[column])
    ax.set_title(column)

# Remove empty subplots (if any)
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# TFT

In [ ]:
import lightning.pytorch as pl
import torch

from lightning.pytorch.tuner import Tuner
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger
from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import EncoderNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE, MAPE, SMAPE #,PoissonLoss load at demand

#from tensorboard import program #cannot call tensorboard from kaggle

In [ ]:
#add extra real feature "month_to_date_return"
# Step 1: Select numeric columns to compute differences
numeric_cols = d_hpr.select_dtypes(include=['float64']).columns

# Step 2: Calculate row-wise cumulative sum for numeric columns
mtd_diff_df = d_hpr.groupby(["Month","Year"], observed = True)[numeric_cols].cumsum()

# Step 3: Add differences to the original DataFrame (optional)
# Prefix new columns with 'mtd_'
d_hpr[[f'mtd_{col}' for col in numeric_cols]] = mtd_diff_df

# Step 4: Debug - Check for infinite or NaN values in month_to_date_return
print("Any infinite values in differences:", mtd_diff_df.apply(np.isinf).any().any())
print("Any NaN values in differences:", mtd_diff_df.isna().any().any())
print(f"\n {d_hpr.head(5)}")

In [ ]:
# add time index for TimeSeriesDataSet use
d_hpr["time_idx"] = range(len(d_hpr))
#becasue of the non-business days, data["time_idx"] = (data["date"] - data["date"].min()).dt.days will not create consecutive int with +1 step

# add a dummy group idx for TimeSeriesDataSet use
d_hpr["dummy_group"] = 1

print(f"the max of time_idx: {d_hpr['time_idx'].min()}")
print(f"the min of time_idx: {d_hpr['time_idx'].max()}")
print(f"the length of time_idx: {len(d_hpr['time_idx'])}")
print(f"the data type is : {d_hpr['time_idx'].dtypes}")

In [ ]:
print(f"Any infinite values in differences: {d_hpr.isin([np.inf, -np.inf]).any().any()}")
print(f"Any NaN values in differences: {d_hpr.isna().any().any()} \n")
print(f"the prepared data for final processing {d_hpr.shape}: \n {d_hpr.head(5)} \n")
print(f"the data statistics: \n {d_hpr.describe()} \n")
print(f"the data types of each col: \n {d_hpr.dtypes}")

* ***prepare dataloader for pyotrch-forecasting***

In [ ]:
#dataset slicing
max_prediction_length = 5
max_encoder_length = 20

#training_cutoff = d_hpr["time_idx"].max() - 251*2 #hide out the last 2 years for val and test
#training_data = d_hpr[lambda x: x.time_idx < training_cutoff]
#validation_data = d_hpr[lambda x: (x.time_idx >= training_cutoff) & (x.time_idx < training_cutoff + 251)]
#test_data = d_hpr[lambda x: x.time_idx >= training_cutoff + 251]

training_data = d_hpr[d_hpr["Year"].isin(["2020", "2021", "2022"])]
validation_data = d_hpr[d_hpr["Year"].isin(["2023"])]
test_data = d_hpr[d_hpr["Year"].isin(["2024", "2025"])]

print(f"""
train data shape: {training_data.shape}
val data shape: {validation_data.shape}
test data shape: {test_data.shape}
""")

print(f"Any infinite values in train data: {training_data.isin([np.inf, -np.inf]).any().any()}")
print(f"Any infinite values in val data: {validation_data.isin([np.inf, -np.inf]).any().any()}")
print(f"Any infinite values in test data: {test_data.isin([np.inf, -np.inf]).any().any()} \n")
print(f"Any NaN values in train data: {training_data.isna().any().any()}")
print(f"Any NaN values in val data: {validation_data.isna().any().any()}")
print(f"Any NaN values in test data: {test_data.isna().any().any()}")

In [ ]:
training = TimeSeriesDataSet(
    training_data,
    time_idx="time_idx",
    target="diff_Close",
    group_ids=["dummy_group"],
    max_encoder_length = max_encoder_length,
    max_prediction_length = max_prediction_length,
    time_varying_known_categoricals = ['Day', 'Weekday', 'Week', 'Month'], #"Year" could cause issue, since it is not consistant over val, test dataset, 
    #it can be put in "time_varying_unknown_categoricals".  
    time_varying_unknown_reals = ["diff_Open", "diff_High", "diff_Low", "diff_Close", "diff_Volume", "mtd_diff_Open", "mtd_diff_High", "mtd_diff_Low", 
                               "mtd_diff_Close", "mtd_diff_Volume"],
    target_normalizer = EncoderNormalizer() #transformation="softplus")
)

validation = TimeSeriesDataSet.from_dataset(
    training, 
    validation_data #keep the predict default = False to make all the data will be included by a sliding window
)

test = TimeSeriesDataSet.from_dataset(
    training, 
    test_data
)

# create dataloaders
train_dataloader = training.to_dataloader(
    train = False, #with the train = False, a encoder + prediction sized window will slide through the whole dataset
    batch_size = 9999, #this arg actually controls the batch dimension of the dataset(number of batches), instead of the number of samples in each batch. 
    #and when it >> whole data length during the window sliding, it will automatically = len(dataset) - window size + 1, the total number of window can be fitted 
)
val_dataloader = validation.to_dataloader(
    train = False, batch_size = 9999
)
test_dataloader = test.to_dataloader(
    train = False, batch_size = 9999
)

* ***prepare pl trainer and model for hyperparameter tunning and model fitting***

In [ ]:
pl.seed_everything(666)

lr_tune_logger = TensorBoardLogger(save_dir = "", version = "lr")  # logging results to the current pwd under dir "lr"
lr_early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=10, mode="min"
)

#set up trainer for tunning
lr_trainer = pl.Trainer(
    accelerator = "auto",
    devices = "auto",
    gradient_clip_val = 0.1,
    #fast_dev_run=True,  # comment in for debugging, only 1 training and 1 validation batch to run
    callbacks=[lr_early_stop_callback],
    logger = lr_tune_logger,  
)

#set up tft for tunning 
tft_tuner = TemporalFusionTransformer.from_dataset(
    training,
    # dummy lr required for the following lr_finder initiation
    learning_rate = 0.06,
    hidden_size = 8,  # most important hyperparameter apart from learning rate
    lstm_layers = 2, 
    # number of attention heads. Set to up to 4 for large datasets
    attention_head_size = 2,
    dropout = 0.1,  # between 0.1 and 0.3 are good values
    loss = QuantileLoss(),
    reduce_on_plateau_patience = 100,
)
print(f"Number of parameters in network: {tft_tuner.size() / 1e3:.3f}k")

In [ ]:
# learning rate optimization
lr_tuner = Tuner(lr_trainer).lr_find(
    tft_tuner,
    train_dataloaders = train_dataloader,
    val_dataloaders = val_dataloader,
    max_lr = 0.1,
    min_lr = 1e-6,
)

print(f"suggested learning rate: {lr_tuner.suggestion()}")
fig = lr_tuner.plot(show=True, suggest=True)
fig.show()

In [ ]:
#set up trainer for training
logger = TensorBoardLogger(save_dir = "", version = "train")  # logging results to the current pwd
early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=10, mode="min"
)

trainer = pl.Trainer(
    accelerator = "auto",
    devices = "auto",
    gradient_clip_val = 0.1,
    #fast_dev_run=True,  # comment in for debugging, only 1 training and 1 validation batch to run
    callbacks=[early_stop_callback],
    logger = logger,
)

#set up tft for taining
tft = TemporalFusionTransformer.from_dataset(
    training,
    # not meaningful for finding the learning rate but otherwise very important
    learning_rate = lr_tuner.suggestion(),
    hidden_size = 8,  # most important hyperparameter apart from learning rate
    lstm_layers = 2, 
    # number of attention heads. Set to up to 4 for large datasets
    attention_head_size = 2,
    dropout = 0.1,  # between 0.1 and 0.3 are good values
    loss = QuantileLoss(),
    #log_interval = 10, # uncomment for learning rate finder and otherwise, e.g. to 10 for logging every 10 batches 
    reduce_on_plateau_patience = 100,
)
print(f"Number of parameters in network: {tft.size() / 1e3:.3f}k") 

In [ ]:
# fit network
trainer.fit(
    tft,
    train_dataloaders = train_dataloader,
    val_dataloaders = val_dataloader
)

In [ ]:
#open tensorboard to check the results, does not work on kaggle?
tb = program.TensorBoard()
tb.configure(argv=[None, '--logdir', 'lightning_logs', '--port', '6006'])
print("TensorBoard running at http://localhost:6006/")
tb.main()

In [ ]:
#load the best model according to the validation loss from training log
best_model_path = trainer.checkpoint_callback.best_model_path
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

In [ ]:
#check the metrics on val_dataset
trainer.validate(best_tft, dataloaders=val_dataloader)#, ckpt_path=best_model_path)
#the "ckpt_path" argument is not necessary, but it is good practice to load the best model

In [ ]:
#exam outliers of MAPE
val_results = best_tft.predict(val_dataloader, return_y = True)
apes = torch.abs((val_results.output - val_results.y[0]) / val_results.y[0])
apes.mean() #it is reasonable to suspect there is 0s in y

In [ ]:
torch.sum(val_results.y[0] == 0) #so there is 5 x 0s

In [ ]:
print(f"MAPE by metric function: {MAPE()(val_results.output, val_results.y[0])}")
#the reason of the val metric does not raise inf is, the function has logic designed to handle 0s by adding "1e-8" from the source code
_apes = torch.abs((val_results.output - val_results.y[0]) / (1e-8 + val_results.y[0]))
print(f"MAPE by hands: {_apes.mean()}") #proved the above statement

In [ ]:
#predict on test set
test_pred_x_raw = best_tft.predict(test_dataloader, mode="raw", return_x=True, return_y=True)
test_pred = test_pred_x_raw#.cpu() #move the results from GPU back to CPU

In [ ]:
#evaluation on test set
trainer.test(best_tft, dataloaders=test_dataloader)#, ckpt_path=best_model_path)

In [ ]:
pred_time_idx = test_pred.x["decoder_time_idx"].reshape(-1, 1)
pred = test_pred.output.prediction.reshape(-1, 7)
pred = torch.cat((pred_time_idx, pred), dim=1) #add the time index of the predictions as col=0 for solving the overlap time idx 

In [ ]:
#prediction vs actual plotting function
def plot_single_timeseries_quantile_predictions(tensor1_preds, tensor2_actuals,
                                                quantile_levels=None, quantile_labels=None,
                                                title="Time Series Quantile Prediction"):
    """
    Plots quantile predictions (tensor1_preds) against actual values (tensor2_actuals)
    for a single time series.

    Args:
        tensor1_preds (np.ndarray): Quantile predictions. Shape: [time, features].
                                    Features are assumed to be sorted quantiles.
                                    Example for 7 features: [q0.02, q0.1, q0.25, q0.5, q0.75, q0.9, q0.98]
        tensor2_actuals (np.ndarray): Actual values. Shape: [time].
        quantile_levels (list of float, optional): The actual quantile levels corresponding
                                                   to the features in tensor1_preds.
                                                   Used for generating accurate default labels.
        quantile_labels (list of str, optional): Labels for the prediction intervals.
                                                 If None, default labels will be generated.
                                                 Order should correspond to outermost to innermost interval.
        title (str, optional): The title for the plot.
    """
    if tensor1_preds.ndim != 2:
        raise ValueError("tensor1_preds (predictions) must be 2D [time, features].")
    if tensor2_actuals.ndim != 1:
        raise ValueError("tensor2_actuals (actuals) must be 1D [time].")
    if tensor1_preds.shape[0] != tensor2_actuals.shape[0]:
        raise ValueError("Time dimension of tensor1_preds and tensor2_actuals must match.")

    num_time_steps = tensor1_preds.shape[0]
    num_features = tensor1_preds.shape[1]

    if num_features % 2 == 0:
        raise ValueError("Number of features in tensor1_preds must be odd to have a central median.")
    
    median_index = num_features // 2

    # Define quantile labels if not provided
    if quantile_labels is None:
        if num_features == 7 and quantile_levels and len(quantile_levels) == 7:
            # Generate labels based on provided quantile_levels
            # Assumes pairs are (0,6), (1,5), (2,4) for features
            pi1_lower, pi1_upper = quantile_levels[0], quantile_levels[6]
            pi2_lower, pi2_upper = quantile_levels[1], quantile_levels[5]
            pi3_lower, pi3_upper = quantile_levels[2], quantile_levels[4]
            
            quantile_labels = [
                f"{(pi1_upper - pi1_lower) * 100:.0f}% PI ({pi1_lower:.2f}-{pi1_upper:.2f})", # Outermost
                f"{(pi2_upper - pi2_lower) * 100:.0f}% PI ({pi2_lower:.2f}-{pi2_upper:.2f})", # Middle
                f"{(pi3_upper - pi3_lower) * 100:.0f}% PI ({pi3_lower:.2f}-{pi3_upper:.2f})"  # Innermost
            ]
        elif num_features == 7: # Default for 7 features if specific levels not given
             quantile_labels = ["96% PI (e.g., 0.02-0.98)", "80% PI (e.g., 0.1-0.9)", "50% PI (e.g., 0.25-0.75)"]
        else:
            # Fallback for a different number of features
            quantile_labels = [f"Interval {i+1}" for i in range(num_features // 2)]
    
    if len(quantile_labels) != num_features // 2:
        raise ValueError(f"Expected {num_features // 2} quantile labels, but got {len(quantile_labels)}.")

    # Set a nice seaborn style
    sns.set_theme(style="whitegrid")

    fig, ax = plt.subplots(figsize=(12, 6)) # Single plot

    time_indices = np.arange(num_time_steps)

    # Plot actual values
    ax.plot(time_indices, tensor2_actuals, label="Actual", color="black", marker='o', linestyle='-', zorder=num_features//2 + 2)

    # Plot median prediction (middle feature)
    median_prediction = tensor1_preds[:, median_index]
    median_label_text = f"Median ({quantile_levels[median_index]:.2f}Q)" if quantile_levels and median_index < len(quantile_levels) else "Median Prediction"
    ax.plot(time_indices, median_prediction, label=median_label_text, color="blue", marker='x', linestyle='--', zorder=num_features//2 + 1)

    # Plot prediction intervals
    # Intervals are formed by pairing features from outside in
    # e.g., for 7 features: (feature 0, feature 6), (feature 1, feature 5), (feature 2, feature 4)
    # Colors for intervals - from lighter to darker for better visual hierarchy
    # Using a list of distinct, visually pleasing colors for intervals
    interval_palette = sns.color_palette("Blues", n_colors=num_features // 2 + 2) # Get a few shades
    
    # The quantile_labels should be ordered from outermost to innermost.
    # The loop for j goes from 0 (outermost interval) to (num_features // 2 - 1) (innermost interval).
    for j in range(num_features // 2):
        lower_quantile_idx = j
        upper_quantile_idx = num_features - 1 - j
        
        # Assign colors such that the widest interval is lightest, narrowest is darkest within the theme
        # So, interval_colors[j] means the j-th interval (0 = outermost) gets a progressively darker shade.
        # The label quantile_labels[j] should correspond to this j-th interval.
        ax.fill_between(time_indices, tensor1_preds[:, lower_quantile_idx], tensor1_preds[:, upper_quantile_idx],
                        color=interval_palette[j], # interval_palette[0] is light, interval_palette[num_features//2 -1] is darker
                        alpha=0.3 + (j * 0.1), # Alpha can also increase for inner bands if desired
                        label=quantile_labels[j], 
                        zorder=j+1)


    ax.set_title(title)
    ax.set_xlabel("Time Step")
    ax.set_ylabel("Value")

    if num_time_steps <= 20: # Show markers if not too many time steps
        ax.set_xticks(time_indices)
    else: # Otherwise, let matplotlib decide tick locations for readability
        pass

    # Add legend
    handles, labels = ax.get_legend_handles_labels()
    # Custom sort order for legend: Actual, Median, then PIs (already ordered by plotting)
    # If specific order is needed and not achieved by plotting order:
    # order_preference = ["Actual", median_label_text] + quantile_labels
    # sorted_legend = sorted(zip(handles, labels), key=lambda x: order_preference.index(x[1]) if x[1] in order_preference else float('inf'))
    # handles = [h for h, l in sorted_legend]
    # labels = [l for h, l in sorted_legend]
    ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1.02, 1))


    plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to make space for the legend outside
    plt.show()

In [ ]:
#create a function to get the avg of the duplicate rows, because of the multihorizon prediciton having overlap of prediction time_idx
def average_duplicate_rows(tensor, column_index):
    """
    Calculates the average of other columns for duplicate rows based on values in a specified column.

    Args:
        tensor: A 2D PyTorch tensor.
        column_index: The index of the column used for identifying duplicate rows.

    Returns:
        A new tensor with averaged rows.
    """

    if tensor.ndim != 2:
      raise ValueError("Input tensor must be 2D.")
    
    if column_index >= tensor.shape[1]:
        raise IndexError("column_index out of range")
    
    # Convert to NumPy array for easier manipulation
    tensor_np = tensor.numpy()

    # Get unique values in the specified column and their indices
    unique_values, inverse_indices = torch.unique(tensor[:, column_index], return_inverse=True)
    unique_values = unique_values.numpy()
    inverse_indices = inverse_indices.numpy()

    # Create a dictionary to store rows for each unique value
    grouped_rows = {}
    for i, val in enumerate(inverse_indices):
        if val not in grouped_rows:
          grouped_rows[val] = []
        grouped_rows[val].append(tensor_np[i])

    # Calculate the average of other columns for each group
    averaged_rows = []
    for _, rows in grouped_rows.items():
        rows = torch.tensor(rows) #convert list of rows to tensor
        averaged_row = rows.mean(dim=0)
        averaged_rows.append(averaged_row)
    
    averaged_rows = torch.stack(averaged_rows)

    return averaged_rows

In [ ]:
pred = average_duplicate_rows(pred, 0) #by somehow the "test_pred_x_raw" is not shown on GPU, but the post processing "pred" is on GPU?
test_plot = torch.tensor(test_data[test_data["time_idx"] >= pred[0,0].numpy()]["diff_Close"].values)
pred_plot = pred[:, 1:] #remove the extra time idx from above

In [ ]:
# --- Define Quantile Labels based on SPECIFIC_QUANTILE_LEVELS ---
# These labels will be passed to the plotting function.
SPECIFIC_QUANTILE_LEVELS= [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98] # The quantile levels used in the model

# The function can also generate them if quantile_levels are passed.
# Order: Outermost, Middle, Innermost
pi_outer_label = f"{(SPECIFIC_QUANTILE_LEVELS[6] - SPECIFIC_QUANTILE_LEVELS[0])*100:.0f}% PI ({SPECIFIC_QUANTILE_LEVELS[0]:.2f}-{SPECIFIC_QUANTILE_LEVELS[6]:.2f})"
pi_mid_label = f"{(SPECIFIC_QUANTILE_LEVELS[5] - SPECIFIC_QUANTILE_LEVELS[1])*100:.0f}% PI ({SPECIFIC_QUANTILE_LEVELS[1]:.2f}-{SPECIFIC_QUANTILE_LEVELS[5]:.2f})"
pi_inner_label = f"{(SPECIFIC_QUANTILE_LEVELS[4] - SPECIFIC_QUANTILE_LEVELS[2])*100:.0f}% PI ({SPECIFIC_QUANTILE_LEVELS[2]:.2f}-{SPECIFIC_QUANTILE_LEVELS[4]:.2f})"
    
# The order in custom_labels MUST match the order of plotting fill_between
# which is from outermost to innermost (j=0 to num_features//2 - 1)
custom_interval_labels = [pi_outer_label, pi_mid_label, pi_inner_label]

In [ ]:
# --- Plot the pred_return v true_return ---
print(f"Tensor1 (predictions) shape: {pred_plot.shape}")
print(f"Tensor2 (actuals) shape: {test_plot.shape}")
print(f"Quantile levels being plotted: {SPECIFIC_QUANTILE_LEVELS}")

plot_single_timeseries_quantile_predictions(
    pred_plot,
    test_plot,
    quantile_levels=SPECIFIC_QUANTILE_LEVELS,
    quantile_labels=custom_interval_labels, # Pass the correctly ordered labels
    title="log Return Prediction with Quantiles"
)

In [ ]:
#plot log close predictions, since the target = "diff_Close"
#the true log price from test set
test_date = test_data[test_data["time_idx"] >= (pred[0,0]-1).numpy()][["Date", "time_idx"]] #find the "time_idx" corresponding "Date" to slice the respective "Close" price
#get the extra t_-1 price for the 1st prediction price calculation

test_price = data[data["Date"].isin(test_date.Date)]["Close"] 
test_price = torch.tensor(test_price.values)
test_price = test_price.reshape(test_price.shape[0], 1) #to match the 'pred' shape for broadcasting operation

#construct prediciton log price by logP_t + pred_logR_t = logP_t+1
pred_price = test_price[:test_price.shape[0]-1,] + pred[...,1:]

test_price_plot = test_price[1:,].squeeze(1)

In [ ]:
# --- Plot the pred_price v true_price ---
print(f"Tensor1 (predictions) shape: {pred_price.shape}")
print(f"Tensor2 (actuals) shape: {test_price_plot.shape}")
print(f"Quantile levels being plotted: {SPECIFIC_QUANTILE_LEVELS}")

plot_single_timeseries_quantile_predictions(
    pred_price,
    test_price_plot,
    quantile_levels=SPECIFIC_QUANTILE_LEVELS,
    quantile_labels=custom_interval_labels, # Pass the correctly ordered labels
    title="log Price Prediction with Quantiles"
)